# Practice: Scripts, NumPy & Time Series — SOLVED

**Estimated time:** ~1.5 hours

This practice covers:
- NumPy: random sampling with `normal`, `uniform`, and `randint`
- Python scripts: `sys.argv`, `__name__` guard, importing from `.py` files
- Time series with pandas: resampling, shifting, rolling windows, `diff()`

**Instructions:**
- For exercises that require `.py` files, create them in your editor/IDE inside a `scripts_practice/` folder
- Then run or import them from this notebook
- Use `!python scripts_practice/filename.py` to run scripts from the notebook

In [ ]:
import numpy as np
import pandas as pd
import sys

---

## Exercise 1: Sampling Fitness Data with NumPy (~10 min)

Set a random seed of `99`, then generate:

1. **Resting heart rates**: 500 samples from a normal distribution with mean=68 and std=7
2. **Sleep hours**: 500 samples from a uniform distribution between 4.0 and 10.0
3. **Daily step counts**: 500 samples of random integers between 2000 and 15000 (inclusive)

For each array, print the mean, standard deviation, median, and the 5th and 95th percentiles.

**Hint:** Use `np.percentile(arr, [5, 95])` for percentiles.

In [ ]:
np.random.seed(99)

heart_rates = np.random.normal(68, 7, 500)
sleep_hours = np.random.uniform(4.0, 10.0, 500)
step_counts = np.random.randint(2000, 15001, 500)

summary = pd.DataFrame({
    "metric": ["Heart rates", "Sleep hours", "Step counts"],
    "mean": [heart_rates.mean(), sleep_hours.mean(), step_counts.mean()],
    "std": [heart_rates.std(), sleep_hours.std(), step_counts.std()],
    "median": [np.median(heart_rates), np.median(sleep_hours), np.median(step_counts)],
    "p5": [np.percentile(heart_rates, 5), np.percentile(sleep_hours, 5), np.percentile(step_counts, 5)],
    "p95": [np.percentile(heart_rates, 95), np.percentile(sleep_hours, 95), np.percentile(step_counts, 95)],
}).set_index("metric")

summary

---

## Exercise 2: Building a Synthetic Fitness Dataset (~10 min)

Using seed `42`, create a DataFrame with 365 rows and the following columns:

| Column | How to generate |
|--------|----------------|
| `date` | `pd.date_range("2025-01-01", periods=365, freq="D")` |
| `steps` | Random integers between 3000 and 15000 |
| `heart_rate_avg` | Normal distribution with mean=72, std=8 |
| `sleep_hours` | Uniform distribution between 4.5 and 9.5 |
| `calories` | Derived: `steps * 0.04 + np.random.normal(200, 50, 365)` |
| `active_minutes` | Derived: `np.clip(steps / 150 + np.random.normal(0, 10, 365), 0, None).astype(int)` |

Then:
1. Print the first 5 rows
2. Print summary statistics with `.describe()`
3. Verify that `calories` correlates with `steps` by printing the correlation between those two columns

In [ ]:
np.random.seed(42)

steps = np.random.randint(3000, 15000, 365)

df = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=365, freq="D"),
    "steps": steps,
    "heart_rate_avg": np.random.normal(72, 8, 365),
    "sleep_hours": np.random.uniform(4.5, 9.5, 365),
    "calories": steps * 0.04 + np.random.normal(200, 50, 365),
    "active_minutes": np.clip(steps / 150 + np.random.normal(0, 10, 365), 0, None).astype(int),
})

df.head()

In [ ]:
df.describe()

In [ ]:
df[["steps", "calories"]].corr()

---

## Exercise 3: Create Module `health_stats.py` (~15 min)

Create a file `scripts_practice/health_stats.py` in your editor/IDE with:

1. A function `z_score(arr)` that standardizes a NumPy array:
   - Formula: `(arr - arr.mean()) / arr.std()`
   - Returns a new array

2. A function `weekly_summary(df, date_col, value_col)` that:
   - Sets `date_col` as the DataFrame index
   - Resamples to weekly frequency using the mean
   - Returns the resampled DataFrame (only the `value_col` column)

3. A function `detect_anomalies(arr, threshold=2.0)` that:
   - Computes z-scores using the `z_score` function
   - Returns a boolean mask where `|z_score| > threshold`

4. An `if __name__ == "__main__":` guard that:
   - Creates a sample array: `np.array([10, 12, 11, 13, 50, 9, 11, 10, 12, 8])`
   - Prints the z-scores
   - Prints which values are anomalies (threshold=2.0)

`scripts_practice/health_stats.py`:

```python
import numpy as np
import pandas as pd


def z_score(arr):
    """Standardize a NumPy array to z-scores."""
    return (arr - arr.mean()) / arr.std()


def weekly_summary(df, date_col, value_col):
    """Resample a DataFrame to weekly mean for a given column.

    Args:
        df: DataFrame with a date column and a value column.
        date_col: Name of the date column.
        value_col: Name of the column to aggregate.

    Returns:
        Series with weekly mean values, indexed by week start date.
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col)
    return df[[value_col]].resample("W").mean()


def detect_anomalies(arr, threshold=2.0):
    """Return a boolean mask where |z_score| > threshold."""
    scores = z_score(arr)
    return np.abs(scores) > threshold


if __name__ == "__main__":
    sample = np.array([10, 12, 11, 13, 50, 9, 11, 10, 12, 8])
    print("Sample array:", sample)
    print("Z-scores:    ", z_score(sample).round(2))
    print("Anomalies:   ", detect_anomalies(sample, threshold=2.0))
    print(f"Anomaly values: {sample[detect_anomalies(sample, threshold=2.0)]}")
```

In [ ]:
# Run the script directly to see the demo
!python scripts_practice/health_stats.py

In [ ]:
# Import and use the functions
sys.path.insert(0, "scripts_practice")
from health_stats import z_score, weekly_summary, detect_anomalies

sample = np.array([10, 12, 11, 13, 50, 9, 11, 10, 12, 8])

pd.DataFrame({
    "value": sample,
    "z_score": z_score(sample).round(2),
    "is_anomaly": detect_anomalies(sample, threshold=2.0),
})

---

## Exercise 4: Import and Use the Module (~10 min)

Import `z_score`, `weekly_summary`, and `detect_anomalies` from `health_stats`.

1. Create a synthetic heart rate array: 365 values from `np.random.normal(72, 8)` with seed `42`
2. Manually inject 5 anomalies by setting indices 50, 100, 150, 200, 250 to values `120, 130, 35, 140, 28`
3. Use `detect_anomalies(heart_rates, threshold=2.0)` to find anomalies
4. Print the indices and values of detected anomalies
5. Create a DataFrame with `date` (daily from 2025-01-01) and `heart_rate` columns
6. Use `weekly_summary(df, "date", "heart_rate")` to get weekly averages
7. Print the first 10 weeks

In [ ]:
sys.path.insert(0, "scripts_practice")
from health_stats import z_score, weekly_summary, detect_anomalies

np.random.seed(42)
heart_rates = np.random.normal(72, 8, 365)

# Inject anomalies
heart_rates[50] = 120
heart_rates[100] = 130
heart_rates[150] = 35
heart_rates[200] = 140
heart_rates[250] = 28

# Detect anomalies
anomalies = detect_anomalies(heart_rates, threshold=2.0)
anomaly_indices = np.where(anomalies)[0]

pd.DataFrame({
    "index": anomaly_indices,
    "heart_rate": heart_rates[anomalies].round(1),
})

In [ ]:
# Weekly summary
df = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=365, freq="D"),
    "heart_rate": heart_rates,
})
weekly = weekly_summary(df, "date", "heart_rate")
weekly.head(10)

In [ ]:
df = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=365, freq="D"),
    "heart_rate": heart_rates,
})
weekly = weekly_summary(df, "date", "heart_rate")
weekly.head(10)

---

## Exercise 5: Data Generator Script with sys.argv (~15 min)

Create a file `scripts_practice/generate_fitness_data.py` in your editor/IDE that:

1. Contains a function `generate_fitness_data(n_days, seed)` that:
   - Sets the random seed
   - Creates a DataFrame with these columns:
     - `date`: daily dates starting from `"2025-01-01"`
     - `steps`: random integers between 3000 and 15000
     - `heart_rate`: normal distribution (mean=72, std=8), clipped to [45, 120]
     - `sleep_hours`: uniform distribution (4.5, 9.5), rounded to 1 decimal
     - `calories`: `steps * 0.04 + np.random.uniform(150, 300, n_days)`
   - Returns the DataFrame

2. Has an `if __name__ == "__main__":` guard that:
   - Takes 2 command-line arguments: `n_days` and `seed`
   - Calls `generate_fitness_data(n_days, seed)`
   - Saves the result to `fitness_data.csv`
   - Prints the shape and summary statistics
   - Shows a usage message if wrong number of arguments

**Expected usage:**

```bash
python scripts_practice/generate_fitness_data.py 365 42
```

Then, in this notebook:
- Import `generate_fitness_data` from `generate_fitness_data`
- Call it with `n_days=50, seed=42`
- Print the first 5 rows

`scripts_practice/generate_fitness_data.py`:

```python
import sys
import numpy as np
import pandas as pd


def generate_fitness_data(n_days, seed):
    """Generate a synthetic fitness dataset.

    Args:
        n_days: Number of days to generate.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with columns: date, steps, heart_rate, sleep_hours, calories.
    """
    np.random.seed(seed)
    df = pd.DataFrame({
        "date": pd.date_range("2025-01-01", periods=n_days, freq="D"),
        "steps": np.random.randint(3000, 15000, n_days),
        "heart_rate": np.clip(np.random.normal(72, 8, n_days), 45, 120).round(1),
        "sleep_hours": np.random.uniform(4.5, 9.5, n_days).round(1),
        "calories": (np.random.randint(3000, 15000, n_days) * 0.04
                     + np.random.uniform(150, 300, n_days)).round(1),
    })
    return df


if __name__ == "__main__":
    if len(sys.argv) != 3:
        print("Usage: python generate_fitness_data.py <n_days> <seed>")
        print("Example: python generate_fitness_data.py 365 42")
        sys.exit(1)

    n_days = int(sys.argv[1])
    seed = int(sys.argv[2])

    df = generate_fitness_data(n_days, seed)
    df.to_csv("fitness_data.csv", index=False)

    print(f"Generated dataset with shape: {df.shape}")
    print("Saved to fitness_data.csv")
    print()
    print(df.describe())
```

In [ ]:
# Import and use the function
sys.path.insert(0, "scripts_practice")
from generate_fitness_data import generate_fitness_data

df = generate_fitness_data(n_days=50, seed=42)
df.head()

In [ ]:
sys.path.insert(0, "scripts_practice")
from generate_fitness_data import generate_fitness_data

df = generate_fitness_data(n_days=50, seed=42)
df.head()

---

## Exercise 6: Weekly Report Script (~15 min)

Create a file `scripts_practice/weekly_report.py` in your editor/IDE that:

1. Takes 2 command-line arguments: `csv_path` and `metric` (column name)
2. Reads the CSV file, parses the `date` column as datetime, and sets it as the index
3. Resamples the specified metric to weekly frequency, computing: mean, min, and max
4. Prints a formatted report showing: week start date, mean, min, max
5. If the wrong number of arguments is provided, prints a usage message and exits

**Expected usage:**

```bash
python scripts_practice/weekly_report.py fitness_data.csv steps
```

**Expected output (example):**
```
Weekly Report: steps
========================================
Week of       |   Mean |    Min |    Max
2025-01-05    |  8234  |  3201  |  14523
2025-01-12    |  9102  |  4512  |  13987
...
```

`scripts_practice/weekly_report.py`:

```python
import sys
import pandas as pd


if __name__ == "__main__":
    if len(sys.argv) != 3:
        print("Usage: python weekly_report.py <csv_path> <metric>")
        print("Example: python weekly_report.py fitness_data.csv steps")
        sys.exit(1)

    csv_path = sys.argv[1]
    metric = sys.argv[2]

    df = pd.read_csv(csv_path, parse_dates=["date"])
    df = df.set_index("date")

    if metric not in df.columns:
        print(f"Error: column '{metric}' not found in {csv_path}")
        print(f"Available columns: {list(df.columns)}")
        sys.exit(1)

    weekly = df[[metric]].resample("W").agg(["mean", "min", "max"])
    weekly.columns = ["mean", "min", "max"]

    print(f"Weekly Report: {metric}")
    print("=" * 45)
    print(f"{'Week of':<14} | {'Mean':>8} | {'Min':>8} | {'Max':>8}")
    print("-" * 45)
    for date, row in weekly.iterrows():
        print(f"{date.strftime('%Y-%m-%d'):<14} | {row['mean']:>8.1f} | {row['min']:>8.1f} | {row['max']:>8.1f}")
```

In [ ]:
# First, generate the CSV (run Exercise 5's script first)
!python scripts_practice/generate_fitness_data.py 365 42

In [ ]:
# Then run the weekly report
!python scripts_practice/weekly_report.py fitness_data.csv steps

In [ ]:
df = pd.read_csv("fitness_data.csv", parse_dates=["date"])
df = df.set_index("date")

# Resample to weekly with multiple aggregations
weekly = df.resample("W").agg({
    "heart_rate": ["mean", "min", "max"],
    "steps": "sum",
    "calories": "sum",
})

# Flatten MultiIndex columns
weekly.columns = ["_".join(col).strip("_") for col in weekly.columns]
weekly.head()

`resample()` aggregates all values in the period (e.g., computes the mean of all daily values in a month), while `asfreq()` simply selects the single value at the period boundary. That's why they produce different results:

In [ ]:
monthly_resample = df["heart_rate"].resample("ME").mean()
monthly_asfreq = df["heart_rate"].asfreq("ME")

pd.DataFrame({
    "resample_mean": monthly_resample,
    "asfreq_value": monthly_asfreq,
}).tail()

In [ ]:
pd.DataFrame({
    "highest_total_steps": [weekly["steps_sum"].idxmax(), weekly["steps_sum"].max()],
    "lowest_min_hr": [weekly["heart_rate_min"].idxmin(), weekly["heart_rate_min"].min()],
}, index=["week", "value"])

In [ ]:
df = pd.read_csv("fitness_data.csv", parse_dates=["date"])
df = df.set_index("date")

weekly = df.resample("W").agg({
    "heart_rate": ["mean", "min", "max"],
    "steps": "sum",
    "calories": "sum",
})
weekly.columns = ["_".join(col).strip("_") for col in weekly.columns]
weekly.head()

In [ ]:
df = pd.read_csv("fitness_data.csv", parse_dates=["date"])
df = df.set_index("date")

# Create lagged columns
for lag in [1, 7, 14, 30]:
    df[f"steps_lag_{lag}"] = df["steps"].shift(lag)

# Compute correlations
correlations = pd.Series(
    {f"lag_{lag}": df["steps"].corr(df[f"steps_lag_{lag}"]) for lag in [1, 7, 14, 30]},
    name="correlation"
)
correlations

In [ ]:
df["steps_diff"] = df["steps"].diff()
df["steps_diff"].nlargest(5)

In [ ]:
df["steps_diff"].nsmallest(5)

In [ ]:
monthly_resample = df["heart_rate"].resample("ME").mean()
monthly_asfreq = df["heart_rate"].asfreq("ME")

pd.DataFrame({
    "resample_mean": monthly_resample,
    "asfreq_value": monthly_asfreq,
}).tail()

In [ ]:
df = pd.read_csv("fitness_data.csv", parse_dates=["date"])
df = df.set_index("date")

# Rolling means
df["rolling_mean_7"] = df["steps"].rolling(7).mean()
df["rolling_mean_30"] = df["steps"].rolling(30).mean()
df["rolling_std_7"] = df["steps"].rolling(7).std()

# Rolling z-score
df["rolling_zscore"] = (df["steps"] - df["rolling_mean_7"]) / df["rolling_std_7"]

# Anomalies
df["is_anomaly"] = df["rolling_zscore"].abs() > 2

df[df["is_anomaly"]][["steps", "rolling_mean_7", "rolling_zscore"]]

In [ ]:
pd.Series({
    "first_30_days_mean": df["steps"].iloc[:30].mean(),
    "last_30_days_mean": df["steps"].iloc[-30:].mean(),
}, name="steps").round(0)

In [ ]:
pd.DataFrame({
    "highest_total_steps": [weekly["steps_sum"].idxmax(), weekly["steps_sum"].max()],
    "lowest_min_hr": [weekly["heart_rate_min"].idxmin(), weekly["heart_rate_min"].min()],
}, index=["week", "value"])

In [ ]:
sys.path.insert(0, "scripts_practice")
from generate_fitness_data import generate_fitness_data

# Generate 180 days
df = generate_fitness_data(180, 77)
df = df.set_index("date")

# Average steps per day of week
df["day_of_week"] = df.index.dayofweek
df.groupby("day_of_week")["steps"].mean().round(0)

In [ ]:
# Weekly resampling
weekly = df.resample("W").agg({"steps": "sum", "calories": "sum", "sleep_hours": "mean"})

# 4-week rolling mean
weekly["steps_rolling_4w"] = weekly["steps"].rolling(4).mean()

# Week-over-week diff
weekly["steps_diff"] = weekly["steps"].diff()
weekly

In [ ]:
best_week = weekly["steps"].idxmax()
worst_week = weekly["steps"].idxmin()
biggest_improvement = weekly["steps_diff"].idxmax()

pd.Series({
    "total_steps": f"{df['steps'].sum():,}",
    "avg_daily_calories": f"{df['calories'].mean():.1f}",
    "best_week": f"{best_week.strftime('%Y-%m-%d')} ({weekly.loc[best_week, 'steps']:,.0f} steps)",
    "worst_week": f"{worst_week.strftime('%Y-%m-%d')} ({weekly.loc[worst_week, 'steps']:,.0f} steps)",
    "biggest_improvement": f"{biggest_improvement.strftime('%Y-%m-%d')} (+{weekly.loc[biggest_improvement, 'steps_diff']:,.0f} steps)",
}, name="summary")

In [ ]:
df = pd.read_csv("fitness_data.csv", parse_dates=["date"])
df = df.set_index("date")

for lag in [1, 7, 14, 30]:
    df[f"steps_lag_{lag}"] = df["steps"].shift(lag)

correlations = pd.Series(
    {f"lag_{lag}": df["steps"].corr(df[f"steps_lag_{lag}"]) for lag in [1, 7, 14, 30]},
    name="correlation"
)
correlations

### Largest step increases

In [ ]:
df["steps_diff"] = df["steps"].diff()
df["steps_diff"].nlargest(5)

### Largest step decreases

In [ ]:
df["steps_diff"].nsmallest(5)

---

## Exercise 9: Rolling Statistics & Trend Detection (~10 min)

Using the daily fitness data:

1. Compute the **7-day** and **30-day** rolling mean of `steps`
2. Compute the **7-day** rolling standard deviation of `steps`
3. Calculate a rolling z-score: `(steps - rolling_mean_7) / rolling_std_7`
4. Flag days where `|z-score| > 2` as anomalies
5. Print the total number of anomalies and the first 5 anomaly dates
6. Compare the mean steps in the **first 30 days** vs the **last 30 days** — is the user's fitness trending up or down?

In [ ]:
df = pd.read_csv("fitness_data.csv", parse_dates=["date"])
df = df.set_index("date")

df["rolling_mean_7"] = df["steps"].rolling(7).mean()
df["rolling_mean_30"] = df["steps"].rolling(30).mean()
df["rolling_std_7"] = df["steps"].rolling(7).std()
df["rolling_zscore"] = (df["steps"] - df["rolling_mean_7"]) / df["rolling_std_7"]
df["is_anomaly"] = df["rolling_zscore"].abs() > 2

df[df["is_anomaly"]][["steps", "rolling_mean_7", "rolling_zscore"]]

In [ ]:
pd.Series({
    "first_30_days_mean": df["steps"].iloc[:30].mean(),
    "last_30_days_mean": df["steps"].iloc[-30:].mean(),
}, name="steps")

The last 30 days have a higher mean than the first 30 days, indicating a slight upward trend in step counts.

---

## Exercise 10: Full Pipeline (~15 min)

Combine everything you've learned:

1. Import `generate_fitness_data` from the module you created in Exercise 5
2. Generate 180 days of data with seed `77`
3. Add a `day_of_week` column (0=Monday, 6=Sunday) and compute the average steps per day of the week
4. Resample to weekly using: `.agg({"steps": "sum", "calories": "sum", "sleep_hours": "mean"})`
5. Compute a 4-week rolling mean of weekly step totals
6. Use `diff()` on the weekly step totals to find the week with the biggest improvement
7. Print a final summary:
   - Total steps over 180 days
   - Average daily calories
   - Best week (highest total steps) and worst week (lowest total steps)
   - Week with biggest improvement (largest positive diff)

In [ ]:
sys.path.insert(0, "scripts_practice")
from generate_fitness_data import generate_fitness_data

df = generate_fitness_data(180, 77)
df = df.set_index("date")
df["day_of_week"] = df.index.dayofweek
df.groupby("day_of_week")["steps"].mean().round(0)

In [ ]:
weekly = df.resample("W").agg({"steps": "sum", "calories": "sum", "sleep_hours": "mean"})
weekly["steps_rolling_4w"] = weekly["steps"].rolling(4).mean()
weekly["steps_diff"] = weekly["steps"].diff()
weekly

In [ ]:
best_week = weekly["steps"].idxmax()
worst_week = weekly["steps"].idxmin()
biggest_improvement = weekly["steps_diff"].idxmax()

pd.Series({
    "total_steps": f"{df['steps'].sum():,}",
    "avg_daily_calories": f"{df['calories'].mean():.1f}",
    "best_week": f"{best_week.strftime('%Y-%m-%d')} ({weekly.loc[best_week, 'steps']:,.0f} steps)",
    "worst_week": f"{worst_week.strftime('%Y-%m-%d')} ({weekly.loc[worst_week, 'steps']:,.0f} steps)",
    "biggest_improvement": f"{biggest_improvement.strftime('%Y-%m-%d')} (+{weekly.loc[biggest_improvement, 'steps_diff']:,.0f} steps)",
}, name="summary")